# Exploration — Motifs de sanction ACPR

Objectif : reconstruire le texte complet de chaque décision (recollage des chunks),
observer la fréquence des segments de motif bruts, et lire le texte des décisions
à motif vide ou particulier pour leur attribuer un motif par analogie.

Prérequis venv local : `google-cloud-bigquery`, `pandas`, `jupyter`.


In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

# ── Config — ajuste le chemin si ton venv training n'est pas au même niveau ──
GCP_PROJECT_ID = "gen-lang-client-0989575872"
GCP_SA_KEY_PATH = Path("../../../gcp_sa_banque.json")  # racine du MVP banque-de-france
BQ_DATASET = "banque_de_france_veille"
BQ_TABLE = "articles_bruts"

# EXPORT_DIR = Path("exploration")
# EXPORT_DIR.mkdir(exist_ok=True)


In [2]:
# ── Connexion BigQuery ──
credentials = service_account.Credentials.from_service_account_file(
    str(GCP_SA_KEY_PATH),
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
client = bigquery.Client(project=GCP_PROJECT_ID, credentials=credentials)

query = f"""
    SELECT id, title, date, content, metadata
    FROM `{GCP_PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}`
    WHERE source = 'acpr_decision'
"""

df = client.query(query).to_dataframe()
print(f"{len(df)} chunks récupérés")
df.head()


c:\Users\iandr\Documents\EXP\exp 2.0\projet cv\presentation-portefolio\realisations\banque-de-france\training\venv-banque-training\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


1924 chunks récupérés


,id,title,date,content,metadata
0,banque_acpr_2025-01_chunk_0,Décision de la Commission des sanctions n°2025...,2026-05-13,Décision de la Commission des sanctions – proc...,"{""url"": ""https://acpr.banque-france.fr/system/..."
1,banque_acpr_2025-01_chunk_1,Décision de la Commission des sanctions n°2025...,2026-05-13,intérieur de la Commission des sanctions ; La ...,"{""url"": ""https://acpr.banque-france.fr/system/..."
2,banque_acpr_2025-01_chunk_2,Décision de la Commission des sanctions n°2025...,2026-05-13,d’un contrôle sur place par les services du se...,"{""url"": ""https://acpr.banque-france.fr/system/..."
3,banque_acpr_2025-01_chunk_3,Décision de la Commission des sanctions n°2025...,2026-05-13,millions d’euros de primes acquises pour l’ass...,"{""url"": ""https://acpr.banque-france.fr/system/..."
4,banque_acpr_2025-01_chunk_4,Décision de la Commission des sanctions n°2025...,2026-05-13,"et II du livre premier, cet article ne fait pa...","{""url"": ""https://acpr.banque-france.fr/system/..."


In [3]:
# ── Parsing des métadonnées JSON (decision_number, chunk_index, motif) ──
def parse_metadata(row):
    meta = json.loads(row["metadata"])
    return pd.Series({
        "decision_number": meta.get("decision_number", ""),
        "chunk_index": meta.get("chunk_index", 0),
        "motif": meta.get("motif", ""),
    })

meta_df = df.apply(parse_metadata, axis=1)
df = pd.concat([df, meta_df], axis=1)
df.head()


,id,title,date,content,metadata,decision_number,chunk_index,motif
0,banque_acpr_2025-01_chunk_0,Décision de la Commission des sanctions n°2025...,2026-05-13,Décision de la Commission des sanctions – proc...,"{""url"": ""https://acpr.banque-france.fr/system/...",2025-01,0,protection de la clientèle ; respect des oblig...
1,banque_acpr_2025-01_chunk_1,Décision de la Commission des sanctions n°2025...,2026-05-13,intérieur de la Commission des sanctions ; La ...,"{""url"": ""https://acpr.banque-france.fr/system/...",2025-01,1,protection de la clientèle ; respect des oblig...
2,banque_acpr_2025-01_chunk_2,Décision de la Commission des sanctions n°2025...,2026-05-13,d’un contrôle sur place par les services du se...,"{""url"": ""https://acpr.banque-france.fr/system/...",2025-01,2,protection de la clientèle ; respect des oblig...
3,banque_acpr_2025-01_chunk_3,Décision de la Commission des sanctions n°2025...,2026-05-13,millions d’euros de primes acquises pour l’ass...,"{""url"": ""https://acpr.banque-france.fr/system/...",2025-01,3,protection de la clientèle ; respect des oblig...
4,banque_acpr_2025-01_chunk_4,Décision de la Commission des sanctions n°2025...,2026-05-13,"et II du livre premier, cet article ne fait pa...","{""url"": ""https://acpr.banque-france.fr/system/...",2025-01,4,protection de la clientèle ; respect des oblig...


In [4]:
# ── Reconstruction du texte complet par décision (chunks recollés dans l'ordre) ──
def reconstruct_documents(df: pd.DataFrame) -> dict:
    documents = {}
    for decision_number, group in df.groupby("decision_number"):
        ordered = group.sort_values("chunk_index")
        full_text = "\n\n".join(ordered["content"].tolist())
        documents[decision_number] = {
            "title": ordered.iloc[0]["title"],
            "date": ordered.iloc[0]["date"],
            "motif": ordered.iloc[0]["motif"],
            "text": full_text,
            "n_chunks": len(ordered),
        }
    return documents

documents = reconstruct_documents(df)
print(f"{len(documents)} décisions reconstruites")


105 décisions reconstruites


In [5]:
# ── Tableau de fréquence des segments de motif bruts ──
segment_counter = Counter()
for doc in documents.values():
    if not doc["motif"]:
        continue
    segments = [s.strip().lower() for s in doc["motif"].split(";") if s.strip()]
    segment_counter.update(segments)

freq_df = pd.DataFrame(segment_counter.most_common(), columns=["segment", "count"])
freq_df


,segment,count
0,lutte contre le blanchiment des capitaux et le...,56
1,contrôle interne,15
2,établissement de crédit,12
3,établissement de paiement,6
4,protection de la clientèle,5
...,...,...
64,maîtrise des risques,1
65,lutte contre la déshérence,1
66,relations avec le superviseur,1
67,établissement de paiement étranger,1


In [8]:
# ── Décisions à motif vide ou particulier — à lire pour attribution par analogie ──
SPECIAL_CASES = ["2010-01", "2012-04", "2013-01", "2015-02", "2018-06", "2019-02", "2021-05", "2014-04"]

for dn in SPECIAL_CASES:
    doc = documents.get(dn)
    if not doc:
        print(f"⚠️  {dn} — introuvable dans le corpus reconstruit")
        continue
    print("=" * 80)
    print(f"{dn} — {doc['title']} ({doc['date']}) — {doc['n_chunks']} chunks")
    print("=" * 80)
    print(doc["text"][:3000])
    print("\n[...texte tronqué à 3000 caractères, voir export si besoin de plus...]\n")


2010-01 — Décision de la Commission des sanctions n° 2010-01 du 10 janvier 2011 à l'égard de la CAISSE DE CREDIT MUNICIPAL DE TOULON (2011-01-10) — 21 chunks
CAISSE DE CRÉDIT MUNICIPAL DE TOULON (CCMT) ----- Procédure n° 2010-01 Blâme et sanction pécuniaire de 150 000 euros ----- Audition du 16 décembre 2010 Rendue le 10 janvier 2011 ----- AUTORITÉ DE CONTRÔLE PRUDENTIEL COMMISSION DES SANCTIONS _____________ Vu […] ; La Commission des sanctions de l’Autorité de contrôle prudentiel, composée de M. MARTIN LAPRADE, Président, de Mme ALDIGÉ et de MM. CREDOT, FLORIN et ICARD, membres ; Après avoir décidé de faire droit à la demande de la CCMT tendant à ce que l’audience ne soit pas publique et entendu, lors de la séance du 16 décembre 2010 : – M. Jean-Manuel CLEMMER, chargé de la mise en état, en sa présentation du dossier, assisté de M. Clément ROUXEL, juriste du service de la commission des sanctions ; – M. Antoine SAINTOYANT, représentant le directeur général du Trésor, qui a indiqué ne

In [9]:
# ── Export d'une décision en fichier texte, au besoin (lecture confortable) ──
def export_decision(decision_number: str):
    doc = documents.get(decision_number)
    if not doc:
        print(f"⚠️  {decision_number} introuvable")
        return
    path = EXPORT_DIR / f"{decision_number}.txt"
    path.write_text(
        f"{doc['title']}\nDate: {doc['date']}\nMotif brut: {doc['motif']}\n\n{doc['text']}",
        encoding="utf-8",
    )
    print(f"✅ Exporté: {path}")

# Exemple d'usage :
# export_decision("2010-01")


In [6]:
# ── Vue d'ensemble — toutes les décisions avec leur motif brut ──
overview = pd.DataFrame([
    {"decision_number": dn, "title": d["title"], "date": d["date"], "motif": d["motif"], "n_chunks": d["n_chunks"]}
    for dn, d in documents.items()
]).sort_values("decision_number")
overview


,decision_number,title,date,motif,n_chunks
0,2010-01,Décision de la Commission des sanctions n° 201...,2011-01-10,,21
1,2010-02,Décision de la commission des sanctions n° 201...,2011-02-28,capacité professionnelle ; condition d’honorab...,9
2,2010-05,Décision de la Commission des sanctions n° 201...,2011-05-26,lutte contre le blanchiment des capitaux ; con...,17
3,2010-06,Décision de la Commission des sanctions n° 201...,2011-12-16,contrôle des opérations et procédures internes...,46
4,2010-07,Décision de la Commission des sanctions n° 201...,2011-07-15,évaluation des risques ; contrôle interne,10
...,...,...,...,...,...
100,2025-01,Décision de la Commission des sanctions n°2025...,2026-05-13,protection de la clientèle ; respect des oblig...,19
101,263f8ed2f9d7,Décision de la Commission des sanctions du 13 ...,2018-06-13,lutte contre le blanchiment des capitaux et le...,14
102,3996b42891a4,Décision de la Commission des sanctions du 26 ...,2018-07-26,organisme d'assurance ; lutte contre le blanch...,29
103,4edd553ffea1,Décision de la Commission des sanctions du 13 ...,2018-06-13,lutte contre le blanchiment des capitaux et le...,25


In [12]:
# ── Distribution des combinaisons brutes de motifs (chaîne complète) ──
combo_counts = overview[overview["motif"] != ""]["motif"].value_counts()
combo_df = combo_counts.reset_index()
combo_df.columns = ["combinaison_motif", "count"]
combo_df

,combinaison_motif,count
0,lutte contre le blanchiment des capitaux et le...,18
1,établissement de crédit ; lutte contre le blan...,6
2,établissement de paiement ; lutte contre le bl...,5
3,lutte contre le blanchiment des capitaux et le...,4
4,organisme d'assurance ; lutte contre le blanch...,4
5,établissement de monnaie électronique ; lutte ...,4
6,contrats d’assurance sur la vie non réclamés,2
7,intermédiaire d’assurance ; obligation d’infor...,2
8,organisme d’assurance ; gel des avoirs,2
9,capacité professionnelle ; condition d’honorab...,1


In [14]:
# ── Distribution du nombre de segments (griefs) par décision ──
overview["n_segments"] = overview["motif"].apply(
    lambda m: len([s.strip() for s in m.split(";") if s.strip()]) if m else 0
)

print("Min:", overview["n_segments"].min())
print("Max:", overview["n_segments"].max())
print()
print(overview["n_segments"].value_counts().sort_index())

Min: 0
Max: 4

n_segments
0     7
1    30
2    49
3    13
4     6
Name: count, dtype: int64


In [8]:
# ── Normalisation des apostrophes (unifie ' et ’) ──
def normalize_apostrophe(text: str) -> str:
    return text.replace("’", "'")

overview["motif_norm"] = overview["motif"].apply(normalize_apostrophe)

In [16]:
# ── Détection data-driven des segments "autonomes" (jamais seuls = suspects) ──
segments_alone = set()
segments_all = set()

for doc in documents.values():
    if not doc["motif"]:
        continue
    segments = [s.strip().lower() for s in doc["motif"].split(";") if s.strip()]
    segments_all.update(segments)
    if len(segments) == 1:
        segments_alone.add(segments[0])

never_alone = segments_all - segments_alone

print(f"Segments qui apparaissent seuls au moins une fois ({len(segments_alone)}) :")
for s in sorted(segments_alone):
    print(" -", s)

print(f"\nSegments qui n'apparaissent JAMAIS seuls ({len(never_alone)}) — candidats descripteurs :")
for s in sorted(never_alone):
    print(" -", s)

Segments qui apparaissent seuls au moins une fois (12) :
 - abandon des poursuites disciplinaires
 - capital minimum
 - contrats d’assurance sur la vie non réclamés
 - contrats d’assurance sur la vie non réglés
 - contrôle interne
 - droit au compte
 - exigence de fonds propre
 - fonds propres
 - lutte contre le blanchiment des capitaux et le financement du terrorisme
 - lutte contre le blanchiment et le financement du terrorisme
 - respect des obligations d’information et de conseil précontractuels
 - risque de non-conformité

Segments qui n'apparaissent JAMAIS seuls (57) — candidats descripteurs :
 - cabinet de courtage
 - caisse de crédit municipal
 - capacité professionnelle
 - changeur manuel
 - comptes d’épargne salariale
 - condition d’honorabilité
 - condition d’honorabilité des dirigeants
 - contrats d’épargne retraite
 - contrats en déshérence
 - contrôle des opérations et procédures internes
 - devoir de conseil
 - déshérence
 - entreprise d’assurance
 - entreprise d’investi

In [17]:
# ── Recalcul segments_alone / never_alone sur motif normalisé + fréquence totale ──
segments_alone = set()
segment_total_count = Counter()

for motif in overview["motif_norm"]:
    if not motif:
        continue
    segments = [s.strip().lower() for s in motif.split(";") if s.strip()]
    segment_total_count.update(segments)
    if len(segments) == 1:
        segments_alone.add(segments[0])

never_alone_df = pd.DataFrame(
    [(s, c) for s, c in segment_total_count.items() if s not in segments_alone],
    columns=["segment", "total_count"],
).sort_values("total_count", ascending=False)

# never_alone_df

In [ ]:
# ── Réveil de sg-embedding sur OVH + récupération des embeddings des segments ──
import httpx
import asyncio
import time
import numpy as np

OVH_ML_HOST = "51.68.130.23"
ORCHESTRATOR_PORT = 8080
EMBEDDING_PORT = 8004

WAKE_URL = f"http://{OVH_ML_HOST}:{ORCHESTRATOR_PORT}/wake/sg-embedding"
EMBED_URL = f"http://{OVH_ML_HOST}:{EMBEDDING_PORT}/embed"

# 1. Réveil du service
with httpx.Client(timeout=30.0) as client:
    wake_resp = client.post(WAKE_URL)
    wake_resp.raise_for_status()
    print("Wake:", wake_resp.json())

# 2. Attente du démarrage (service on-demand, peut prendre quelques secondes à charger le modèle)
print("Attente du démarrage du service...")
time.sleep(15)

# 3. Appel batch pour les 65 segments — avec retry en cas de service pas encore prêt
def get_embeddings_with_retry(texts, max_retries=5, wait_seconds=5):
    for attempt in range(max_retries):
        try:
            with httpx.Client(timeout=60.0) as client:
                resp = client.post(EMBED_URL, json={"texts": texts})
                resp.raise_for_status()
                return resp.json()["embeddings"]
        except (httpx.ConnectError, httpx.HTTPStatusError) as e:
            print(f"Tentative {attempt + 1}/{max_retries} échouée: {e} — nouvelle tentative dans {wait_seconds}s")
            time.sleep(wait_seconds)
    raise RuntimeError("Service embedding indisponible après plusieurs tentatives")

all_segments_list = sorted(segment_total_count.keys())
segment_embeddings = get_embeddings_with_retry(all_segments_list)

X_segments = np.array(segment_embeddings)
print(f"Embeddings récupérés : {X_segments.shape} ({len(all_segments_list)} segments, {X_segments.shape[1]} dimensions)")

Wake: {'status': 'ok', 'service': 'sg-embedding'}
Attente du démarrage du service...
Embeddings récupérés : (65, 768) (65 segments, 768 dimensions)


In [19]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from collections import defaultdict


# ── Étape 1 (TF-IDF au lieu d'embeddings) : lots équilibrés via KMeans sur TF-IDF ──
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(all_segments_list).toarray()

BATCH_SIZE_TARGET = 12
n_batches = max(1, len(all_segments_list) // BATCH_SIZE_TARGET)

km_batch = KMeans(n_clusters=n_batches, random_state=42, n_init=10)
batch_labels = km_batch.fit_predict(X_tfidf)

batches = defaultdict(list)
for segment, label in zip(all_segments_list, batch_labels):
    batches[label].append(segment)

for label, members in batches.items():
    print(f"Lot {label} ({len(members)} segments)")

# Lot 0 (8 segments)
# Lot 2 (14 segments)
# Lot 4 (29 segments)
# Lot 1 (10 segments)
# Lot 3 (4 segments)

Lot 0 (8 segments)
Lot 2 (14 segments)
Lot 4 (29 segments)
Lot 1 (10 segments)
Lot 3 (4 segments)


In [20]:
# ── Étape 2 : regroupement LLM par lot ──
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
# OLLAMA_MODEL = "gemma3:12b"
# OLLAMA_MODEL = "mistral"
OLLAMA_MODEL = "mistral-nemo"

# ── Regroupement des segments bruts en concepts canoniques (mistral-nemo) ──
# ── Regroupement des segments bruts en concepts canoniques (mistral-nemo, schéma strict) ──
GROUPING_PROMPT = """Tu es un expert en réglementation bancaire et assurantielle française (ACPR).

Voici une liste de segments de texte extraits de motifs de sanction ACPR. Plusieurs segments
désignent en réalité le MÊME concept, juste avec une formulation différente (singulier/pluriel,
synonymes, variantes orthographiques). Ta tâche : regrouper ces segments en concepts canoniques.

Règles :
- Chaque segment de la liste doit appartenir à exactement un groupe.
- Un concept canonique peut n'avoir qu'un seul membre si aucune variante n'existe.
- Donne un nom court et clair à chaque concept canonique (en français, sans majuscules).
- Ne fusionne PAS des concepts différents sous prétexte qu'ils sont liés thématiquement.

Segments à regrouper :
{segments_list}

Exemple de réponse attendue (structure à respecter EXACTEMENT, clés "canonical" et "members") :
{{"groups": [
  {{"canonical": "lutte contre le blanchiment", "members": ["lutte contre le blanchiment des capitaux et le financement du terrorisme", "lutte contre le blanchiment des capitaux"]}},
  {{"canonical": "gel des avoirs", "members": ["gel des avoirs"]}}
]}}
"""

GROUPING_SCHEMA = {
    "type": "object",
    "properties": {
        "groups": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "canonical": {"type": "string"},
                    "members": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["canonical", "members"],
            },
        },
    },
    "required": ["groups"],
}

def group_batch(segments_batch):
    segments_numbered = "\n".join(f"- {s}" for s in segments_batch)
    response = requests.post(OLLAMA_URL, json={
        "model": "mistral-nemo",
        "prompt": GROUPING_PROMPT.format(segments_list=segments_numbered),
        "stream": False,
        "format": GROUPING_SCHEMA,
        "options": {"num_ctx": 4096, "temperature": 0, "num_predict": 2048},
    })
    return json.loads(response.json()["response"])["groups"]

first_pass_groups = []
for label, members in batches.items():
    groups = group_batch(members)
    first_pass_groups.extend(groups)
    print(f"Lot {label} → {len(groups)} concepts")

print(f"\nTotal après 1ère passe : {len(first_pass_groups)} concepts canoniques")

Lot 0 → 5 concepts
Lot 2 → 8 concepts
Lot 4 → 15 concepts
Lot 1 → 4 concepts
Lot 3 → 2 concepts

Total après 1ère passe : 34 concepts canoniques


In [21]:
# ── Vérification : tous les segments sont-ils bien couverts après la 1ère passe ? ──
covered = []
for g in first_pass_groups:
    covered.extend(g["members"])

missing_after_pass1 = set(all_segments_list) - set(covered)
print("Manquants après 1ère passe:", missing_after_pass1)

Manquants après 1ère passe: {'organisation comptable', 'lutte contre la déshérence', "intermédiaire d'assurances", 'cabinet de courtage', "modification de contrats d'assurance", 'contrôle des opérations et procédures internes'}


In [22]:
# ── Réconciliation automatique : les manquants deviennent leurs propres concepts ──
missing_after_pass1 = set(all_segments_list) - set(covered)

for seg in missing_after_pass1:
    first_pass_groups.append({"canonical": seg, "members": [seg]})

print(f"Total après réconciliation : {len(first_pass_groups)} concepts canoniques")

# Vérification finale — plus aucun manquant possible par construction
covered_final = []
for g in first_pass_groups:
    covered_final.extend(g["members"])
assert set(covered_final) == set(all_segments_list), "Il manque encore des segments !"
print("✅ Tous les segments sont couverts")

Total après réconciliation : 40 concepts canoniques
✅ Tous les segments sont couverts


In [23]:
# ── Passe 2 : consolidation finale sur les concepts canoniques de la passe 1 ──
# canonical_names_pass1 = [g["canonical"] for g in first_pass_groups]
# final_groups = group_batch(canonical_names_pass1)  # réutilise la même fonction, un seul appel

pass1_with_context = []
for g in first_pass_groups:
    members_sample = ", ".join(g["members"][:3])  # jusqu'à 3 membres en exemple
    pass1_with_context.append(f"{g['canonical']} (ex: {members_sample})")

final_groups = group_batch(pass1_with_context)

print(f"Après consolidation : {len(final_groups)} concepts canoniques finaux")
pd.DataFrame([
    {"canonical": g["canonical"], "n_members": len(g["members"]), "members": ", ".join(g["members"])}
    for g in final_groups
])

Après consolidation : 34 concepts canoniques finaux


,canonical,n_members,members
0,maîtrise des risques,3,"maîtrise des risques, maîtrise des risques de ..."
1,condition d'honorabilité,3,"condition d'honorabilité des dirigeants, confo..."
2,gel des avoirs,1,gel des avoirs
3,abandon des poursuites disciplinaires,1,abandon des poursuites disciplinaires
4,obligation d'identification,1,obligation d'identification des assurés décédés
5,établissement de crédit,2,"établissement de crédit, établissement de créd..."
6,monnaie électronique,2,"émetteur de monnaie électronique, établissemen..."
7,paiement,2,"établissement de paiement, établissement de pa..."
8,caisse de crédit municipal,1,caisse de crédit municipal
9,institution de prévoyance,1,institution de prévoyance


In [24]:
# ── Reconstruction : segment brut → concept canonique final ──
# D'abord, mapping segment brut -> nom du concept de la passe 1
segment_to_pass1 = {}
for g in first_pass_groups:
    for seg in g["members"]:
        segment_to_pass1[seg] = g["canonical"]

# Puis, mapping nom concept passe 1 -> concept final (passe 2)
pass1_to_final = {}
for g in final_groups:
    for member in g["members"]:
        pass1_to_final[member] = g["canonical"]

# Chaîne complète : segment brut -> concept final
segment_to_final = {}
for seg, pass1_name in segment_to_pass1.items():
    final_name = pass1_to_final.get(pass1_name, pass1_name)  # fallback si absent de la passe 2
    segment_to_final[seg] = final_name

# Vérification
missing_final = set(all_segments_list) - set(segment_to_final.keys())
print("Segments non mappés:", missing_final)

# Vue d'ensemble
final_mapping_df = pd.DataFrame([
    {"segment_brut": seg, "concept_final": final} for seg, final in segment_to_final.items()
]).sort_values("concept_final")
final_mapping_df

final_mapping_df.to_csv("final_mapping_df.csv", sep=";", index=False, encoding="utf-8")

Segments non mappés: set()


In [25]:
# ── Classification ENTITY/GRIEF sur les concepts finaux (un par un, liste courte) ──

# ── Annotation sémantique des segments via Ollama local

# ── Classification par concept, avec sortie par segment membre ──
PER_SEGMENT_PROMPT = """Tu es un expert en réglementation bancaire et assurantielle française (ACPR).

Voici un groupe de segments qui désignent tous le même concept de fond : "{concept}"
Segments du groupe : {members}

Pour CHAQUE segment du groupe, classe-le dans une seule des deux catégories :
- ENTITY : catégorie/statut réglementaire d'entité (ex: établissement de crédit, organisme d'assurance)
- GRIEF : véritable motif de manquement/sanction (ex: lutte contre le blanchiment, gel des avoirs, contrôle interne)

Réponds UNIQUEMENT en JSON strict, un objet par segment, avec EXACTEMENT ces clés :
{{"annotations": [{{"segment": "...", "categorie": "ENTITY" ou "GRIEF", "justification": "..."}}, ...]}}
"""

PER_SEGMENT_SCHEMA = {
    "type": "object",
    "properties": {
        "annotations": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "segment": {"type": "string"},
                    "categorie": {"type": "string"},
                    "justification": {"type": "string"},
                },
                "required": ["segment", "categorie", "justification"],
            },
        },
    },
    "required": ["annotations"],
}

concept_to_members = final_mapping_df.groupby("concept_final")["segment_brut"].apply(list).to_dict()

all_segment_annotations = []
for concept, members in concept_to_members.items():
    members_str = ", ".join(members)
    response = requests.post(OLLAMA_URL, json={
        "model": "mistral-nemo",
        "prompt": PER_SEGMENT_PROMPT.format(concept=concept, members=members_str),
        "stream": False,
        "format": PER_SEGMENT_SCHEMA,
        "options": {"temperature": 0},
    })
    result = json.loads(response.json()["response"])
    for ann in result["annotations"]:
        ann["concept_final"] = concept
    all_segment_annotations.extend(result["annotations"])

all_segment_annotations_df = pd.DataFrame(all_segment_annotations)
print(f"{len(all_segment_annotations_df)} segments annotés (attendu: 65)")
all_segment_annotations_df

66 segments annotés (attendu: 65)


,segment,categorie,justification,concept_final
0,abandon des poursuites disciplinaires,GRIEF,Il s'agit d'un motif de manquement/sanction,abandon des poursuites disciplinaires
1,cabinet de courtage,ENTITY,Un cabinet de courtage est une entité qui agit...,cabinet de courtage
2,caisse de crédit municipal,ENTITY,Les caisses de crédit municipales sont des éta...,caisse de crédit municipal
3,exigence de fonds propre,ENTITY,Cette exigence s'applique aux entités soumises...,capital réglementaire
4,insuffisance de fonds propre,GRIEF,Une insuffisance de fonds propres peut entraîn...,capital réglementaire
...,...,...,...,...
61,risque de non-conformité,GRIEF,Le risque de non-conformité est un motif de ma...,risque
62,succursale bancaire,ENTITY,Une succursale bancaire est une entité qui app...,succursale bancaire
63,contrats d'épargne retraite,ENTITY,Les contrats d'épargne retraite sont des produ...,épargne retraite
64,établissement de crédit,ENTITY,Un établissement de crédit est une entité régl...,établissement de crédit


In [26]:
all_segment_annotations_df["categorie"].value_counts()

categorie
ENTITY    36
GRIEF     30
Name: count, dtype: int64

In [27]:
dupes = all_segment_annotations_df[all_segment_annotations_df["segment"].duplicated(keep=False)]
dupes.sort_values("segment")

,segment,categorie,justification,concept_final


In [28]:
consistency_check = all_segment_annotations_df.groupby("concept_final")["categorie"].nunique()
inconsistent_concepts = consistency_check[consistency_check > 1]
print("Concepts avec categorie incohérente entre leurs membres:")
inconsistent_concepts

Concepts avec categorie incohérente entre leurs membres:


concept_final
capital réglementaire             2
condition d'honorabilité          2
gouvernance                       2
information des assurés           2
intermédiaire en assurance        2
protection des fonds collectés    2
risque                            2
Name: categorie, dtype: int64

In [29]:
# ── Filtrer : ne garder que les vrais segments bruts d'origine ──
all_segment_annotations_df_clean = all_segment_annotations_df[
    all_segment_annotations_df["segment"].isin(all_segments_list)
].copy()

print(f"{len(all_segment_annotations_df_clean)} segments valides (attendu: 65)")

# Vérifier ce qui a été filtré
dropped = all_segment_annotations_df[~all_segment_annotations_df["segment"].isin(all_segments_list)]
print("\nLignes filtrées (probablement des noms de concepts) :")
dropped

63 segments valides (attendu: 65)

Lignes filtrées (probablement des noms de concepts) :


,segment,categorie,justification,concept_final
28,gouvernance,ENTITY,La gouvernance concerne l'ensemble des règles ...,gouvernance
29,contrôle de la conformité,GRIEF,Le contrôle de la conformité est une mesure pr...,gouvernance
62,succursale bancaire,ENTITY,Une succursale bancaire est une entité qui app...,succursale bancaire


In [30]:
# ── Segments réellement manquants après nettoyage ──
missing_segments = set(all_segments_list) - set(all_segment_annotations_df_clean["segment"])
print(f"{len(missing_segments)} segments manquants :")
missing_segments

2 segments manquants :


{'gouvernance, contrôle de la conformité', 'succursale'}

In [31]:
manual_fixes = pd.DataFrame([
    {"segment": "gouvernance, contrôle de la conformité", "categorie": "GRIEF", "justification": "manquement de gouvernance/conformité — ajout manuel", "concept_final": "gouvernance"},
    {"segment": "succursale", "categorie": "ENTITY", "justification": "statut réglementaire d'entité — ajout manuel", "concept_final": "succursale bancaire"},
])

all_segment_annotations_final = pd.concat([
    all_segment_annotations_df_clean[~all_segment_annotations_df_clean["segment"].isin(["gouvernance", "contrôle de la conformité", "succursale bancaire"])],
    manual_fixes,
], ignore_index=True)

print(f"Total final: {len(all_segment_annotations_final)} (attendu: 65)")

Total final: 65 (attendu: 65)


In [32]:
# ── Revérification de cohérence sur la version finale ──
consistency_final = all_segment_annotations_final.groupby("concept_final")["categorie"].agg(lambda x: list(x))
inconsistent_final = consistency_final[consistency_final.apply(lambda x: len(set(x)) > 1)]
inconsistent_final

concept_final
capital réglementaire             [ENTITY, GRIEF, ENTITY, ENTITY]
condition d'honorabilité           [ENTITY, GRIEF, ENTITY, GRIEF]
information des assurés                    [GRIEF, GRIEF, ENTITY]
intermédiaire en assurance                [ENTITY, GRIEF, ENTITY]
protection des fonds collectés            [ENTITY, GRIEF, ENTITY]
risque                                            [ENTITY, GRIEF]
Name: categorie, dtype: object

In [33]:
# ── Résolution manuelle des 8 concepts incohérents (une seule catégorie par concept) ──
concept_final_category = {
    "capital réglementaire": "GRIEF",              # exigences de fonds propres non respectées
    "condition d'honorabilité": "ENTITY",          # prérequis d'agrément, pas un manquement en soi
    # "gouvernance": "GRIEF",                         # manquement de gouvernance/contrôle de la conformité
    "information des assurés": "GRIEF",             # devoir d'information non respecté
    "intermédiaire en assurance": "ENTITY",        # statut réglementaire
    # "paiement": "ENTITY",                           # établissement de paiement, statut d'entité
    "protection des fonds collectés": "GRIEF",     # manquement à l'obligation de protection des fonds
    "risque": "GRIEF",                              # déficience de maîtrise des risques
}

# Appliquer la résolution : override sur tous les segments des concepts concernés
for concept, cat in concept_final_category.items():
    all_segment_annotations_final.loc[
        all_segment_annotations_final["concept_final"] == concept, "categorie"
    ] = cat

# Vérification finale : plus aucune incohérence possible
final_check = all_segment_annotations_final.groupby("concept_final")["categorie"].nunique()
assert (final_check == 1).all(), "Incohérence résiduelle !"
print("✅ Taxonomie ENTITY/GRIEF figée, aucune incohérence")

✅ Taxonomie ENTITY/GRIEF figée, aucune incohérence


In [34]:
# ── Export de la taxonomie validée ──
taxonomy_export = all_segment_annotations_final[["segment", "concept_final", "categorie"]].copy()
taxonomy_export.to_json("taxonomy_mapping.json", orient="records", force_ascii=False, indent=2)
taxonomy_export.to_csv("taxonomy_mapping.csv", index=False)
print(f"Export : {len(taxonomy_export)} segments")

Export : 65 segments


In [11]:
taxonomy_export = pd.read_csv("taxonomy_mapping.csv")
# ── Distribution réelle par grief final (après taxonomie, décomposé) ──
grief_only_mapping = {
    row["segment"]: row["concept_final"]
    for row in taxonomy_export.to_dict("records")
    if row["categorie"] == "GRIEF"
}

def get_griefs_for_decision(motif: str) -> list:
    if not motif:
        return []
    segments = [s.strip().lower() for s in normalize_apostrophe(motif).split(";") if s.strip()]
    griefs = {grief_only_mapping[s] for s in segments if s in grief_only_mapping}
    return list(griefs)

overview["griefs_finaux"] = overview["motif"].apply(get_griefs_for_decision)

# Compter dans combien de décisions chaque grief apparaît
grief_decision_counts = Counter()
for griefs in overview["griefs_finaux"]:
    grief_decision_counts.update(griefs)

grief_freq_df = pd.DataFrame(
    grief_decision_counts.most_common(), columns=["grief_final", "n_decisions"]
)
grief_freq_df



,grief_final,n_decisions
0,lutte contre le blanchiment et le financement ...,58
1,gouvernance,16
2,information des assurés,8
3,protection des fonds collectés,7
4,devoir de conseil,6
5,capital réglementaire,4
6,gel des avoirs,4
7,non-respect de mise en demeure,3
8,contrats en déshérence,3
9,risque,2
